# 4-Fold Corrected Experiments with Composite Loss

This notebook documents the revised experiment run: exact deduplication, 4-fold stratified cross-validation, a validation split inside each fold for Youden tau, and the composite training objective `CE + focal + dice`.

## Runtime Environment

```json
{
  "platform": "Windows-10-10.0.26200-SP0",
  "torch": "2.7.1+cpu",
  "transformers": "4.53.0",
  "cuda_available": false,
  "cuda_device": null
}
```

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd() / 'fourfold_composite_outputs'
summary = pd.read_csv(ROOT / 'fourfold_summary_metrics.csv')
fold = pd.read_csv(ROOT / 'fourfold_fold_metrics.csv')
summary, fold.head()

## Summary Metrics

| disorder        |   dedup_removed |   n_total_dedup |   f1_fold_mean |   f1_fold_std |   oof_f1 |   oof_f1_ci_lo |   oof_f1_ci_hi |   oof_auc |   oof_accuracy |   TP |   FP |   FN |   TN |
|:----------------|----------------:|----------------:|---------------:|--------------:|---------:|---------------:|---------------:|----------:|---------------:|-----:|-----:|-----:|-----:|
| Anger/IED (d1)  |               7 |             297 |         0.9189 |        0.0331 |   0.9191 |         0.8856 |         0.9495 |    0.9784 |         0.9192 |  133 |    8 |   16 |  140 |
| Anxiety (d2)    |              40 |             263 |         0.7472 |        0.0769 |   0.7523 |         0.6983 |         0.8059 |    0.8369 |         0.7529 |   93 |   26 |   39 |  105 |
| Depression (d3) |               9 |             404 |         0.7912 |        0.0769 |   0.797  |         0.7551 |         0.8366 |    0.8981 |         0.797  |  163 |   42 |   40 |  159 |
| NPD (d4)        |               4 |             309 |         0.9677 |        0.0307 |   0.9676 |         0.945  |         0.987  |    0.9959 |         0.9676 |  147 |    1 |    9 |  152 |
| Panic (d5)      |               1 |             221 |         0.8592 |        0.0478 |   0.8597 |         0.81   |         0.9049 |    0.9611 |         0.8597 |   98 |   17 |   14 |   92 |

## Fold-Level Metrics

| disorder        |   fold |   n_train |   n_val |   n_test |   tau_val |     f1 |    auc |   accuracy |   TP |   FP |   FN |   TN |
|:----------------|-------:|----------:|--------:|---------:|----------:|-------:|-------:|-----------:|-----:|-----:|-----:|-----:|
| Anger/IED (d1)  |      1 |       177 |      45 |       75 |    0.9841 | 0.8784 | 0.9879 |     0.88   |   29 |    0 |    9 |   37 |
| Anger/IED (d1)  |      2 |       178 |      45 |       74 |    0.1101 | 0.9187 | 0.9533 |     0.9189 |   36 |    5 |    1 |   32 |
| Anger/IED (d1)  |      3 |       178 |      45 |       74 |    0.5879 | 0.9189 | 0.9774 |     0.9189 |   34 |    3 |    3 |   34 |
| Anger/IED (d1)  |      4 |       178 |      45 |       74 |    0.9876 | 0.9594 | 0.9934 |     0.9595 |   34 |    0 |    3 |   37 |
| Anxiety (d2)    |      1 |       157 |      40 |       66 |    0.3738 | 0.6553 | 0.753  |     0.6667 |   28 |   17 |    5 |   16 |
| Anxiety (d2)    |      2 |       157 |      40 |       66 |    0.8393 | 0.7861 | 0.8705 |     0.7879 |   23 |    4 |   10 |   29 |
| Anxiety (d2)    |      3 |       157 |      40 |       66 |    0.7195 | 0.8302 | 0.9265 |     0.8333 |   23 |    1 |   10 |   32 |
| Anxiety (d2)    |      4 |       158 |      40 |       65 |    0.8014 | 0.7171 | 0.8466 |     0.7231 |   19 |    4 |   14 |   28 |
| Depression (d3) |      1 |       242 |      61 |      101 |    0.967  | 0.8507 | 0.9404 |     0.8515 |   39 |    4 |   11 |   47 |
| Depression (d3) |      2 |       242 |      61 |      101 |    0.9828 | 0.7473 | 0.8894 |     0.7525 |   31 |    5 |   20 |   45 |
| Depression (d3) |      3 |       242 |      61 |      101 |    0.0914 | 0.7054 | 0.8573 |     0.7228 |   49 |   26 |    2 |   24 |
| Depression (d3) |      4 |       242 |      61 |      101 |    0.6736 | 0.8614 | 0.9353 |     0.8614 |   44 |    7 |    7 |   43 |
| NPD (d4)        |      1 |       184 |      47 |       78 |    0.9847 | 0.9226 | 0.9947 |     0.9231 |   33 |    0 |    6 |   39 |
| NPD (d4)        |      2 |       185 |      47 |       77 |    0.469  | 0.987  | 1      |     0.987  |   38 |    0 |    1 |   38 |
| NPD (d4)        |      3 |       185 |      47 |       77 |    0.6203 | 0.974  | 0.9899 |     0.974  |   38 |    1 |    1 |   37 |
| NPD (d4)        |      4 |       185 |      47 |       77 |    0.991  | 0.987  | 0.9993 |     0.987  |   38 |    0 |    1 |   38 |
| Panic (d5)      |      1 |       132 |      33 |       56 |    0.3789 | 0.8555 | 0.9834 |     0.8571 |   27 |    7 |    1 |   21 |
| Panic (d5)      |      2 |       132 |      34 |       55 |    0.8651 | 0.8364 | 0.9643 |     0.8364 |   23 |    4 |    5 |   23 |
| Panic (d5)      |      3 |       132 |      34 |       55 |    0.75   | 0.8178 | 0.9352 |     0.8182 |   24 |    6 |    4 |   21 |
| Panic (d5)      |      4 |       132 |      34 |       55 |    0.9598 | 0.927  | 0.9881 |     0.9273 |   24 |    0 |    4 |   27 |

## 300 dpi Figures

![roc_4fold_anger_300dpi](roc_300dpi/roc_4fold_anger_300dpi.png)

![roc_4fold_anxiety_300dpi](roc_300dpi/roc_4fold_anxiety_300dpi.png)

![roc_4fold_depression_300dpi](roc_300dpi/roc_4fold_depression_300dpi.png)

![roc_4fold_npd_300dpi](roc_300dpi/roc_4fold_npd_300dpi.png)

![roc_4fold_panic_300dpi](roc_300dpi/roc_4fold_panic_300dpi.png)

![loss_4fold_anger_300dpi](loss_300dpi/loss_4fold_anger_300dpi.png)

![loss_4fold_anxiety_300dpi](loss_300dpi/loss_4fold_anxiety_300dpi.png)

![loss_4fold_depression_300dpi](loss_300dpi/loss_4fold_depression_300dpi.png)

![loss_4fold_npd_300dpi](loss_300dpi/loss_4fold_npd_300dpi.png)

![loss_4fold_panic_300dpi](loss_300dpi/loss_4fold_panic_300dpi.png)

![fourfold_oof_f1_bootstrap_ci_300dpi](summary_300dpi/fourfold_oof_f1_bootstrap_ci_300dpi.png)

![fourfold_foldmean_vs_oof_f1_300dpi](summary_300dpi/fourfold_foldmean_vs_oof_f1_300dpi.png)

![fourfold_oof_auc_300dpi](summary_300dpi/fourfold_oof_auc_300dpi.png)

![confusion_4fold_oof_anger_300dpi](summary_300dpi/confusion_4fold_oof_anger_300dpi.png)

![confusion_4fold_oof_anxiety_300dpi](summary_300dpi/confusion_4fold_oof_anxiety_300dpi.png)

![confusion_4fold_oof_depression_300dpi](summary_300dpi/confusion_4fold_oof_depression_300dpi.png)

![confusion_4fold_oof_npd_300dpi](summary_300dpi/confusion_4fold_oof_npd_300dpi.png)

![confusion_4fold_oof_panic_300dpi](summary_300dpi/confusion_4fold_oof_panic_300dpi.png)

## Re-run Command

```powershell
py -3.10 corrected_protocol_4fold_composite_loss.py --epochs 4 --batch-size 4 --bootstrap 2000 --out-dir fourfold_composite_outputs
```

## GPU Run Log Tail

```text

```